### meta-data

In [ ]:
from pathlib import Path

# Ana folder pathini buraya yaz
ROOT_DIR = Path(r"C:\Users\askin\OneDrive\Masaüstü\dataset_clean\sahneler\S8")

# Önce True bırak, ne sileceğini gör.
# Emin olunca False yapıp tekrar çalıştır.
DRY_RUN = False

def main():
    meta_files = sorted(ROOT_DIR.rglob("*_meta.json"))

    print(f"[INFO] Root folder: {ROOT_DIR}")
    print(f"[INFO] Found meta JSON files: {len(meta_files)}")

    if not meta_files:
        print("[INFO] No *_meta.json files found.")
        return

    print("\nFiles to delete:")
    for file_path in meta_files:
        print(file_path)

    if DRY_RUN:
        print("\n[DRY RUN] Hiçbir dosya silinmedi.")
        print("Silmek için DRY_RUN = False yapıp tekrar çalıştır.")
        return

    deleted_count = 0

    for file_path in meta_files:
        try:
            file_path.unlink()
            deleted_count += 1
            print(f"[DELETED] {file_path}")
        except Exception as e:
            print(f"[ERROR] Could not delete {file_path}: {e}")

    print(f"\n[DONE] Deleted meta JSON files: {deleted_count}")

if __name__ == "__main__":
    main()

### pair_checker

In [ ]:
from pathlib import Path

# Ana dataset klasörünü buraya yaz
ROOT_DIR = Path(r"C:\Users\askin\OneDrive\Masaüstü\dataset_clean\sahneler\S2")

# Önce True bırak: sadece rapor verir, silmez.
# Kontrol ettikten sonra False yap.
DRY_RUN = False

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]


def collect_files(root_dir: Path):
    image_files = {}
    json_files = {}

    for path in root_dir.rglob("*"):
        if not path.is_file():
            continue

        suffix = path.suffix.lower()

        # meta jsonları sayma / eşleştirme dışı bırak
        if path.name.endswith("_meta.json"):
            continue

        if suffix in IMAGE_EXTENSIONS:
            image_files[path.with_suffix("").as_posix()] = path

        elif suffix == ".json":
            json_files[path.with_suffix("").as_posix()] = path

    return image_files, json_files


def count_all_relevant_files(root_dir: Path):
    image_count = 0
    json_count = 0

    for path in root_dir.rglob("*"):
        if not path.is_file():
            continue

        suffix = path.suffix.lower()

        if path.name.endswith("_meta.json"):
            continue

        if suffix in IMAGE_EXTENSIONS:
            image_count += 1
        elif suffix == ".json":
            json_count += 1

    return image_count, json_count


def main():
    print(f"[INFO] Root dir: {ROOT_DIR}")

    before_image_count, before_json_count = count_all_relevant_files(ROOT_DIR)

    image_files, json_files = collect_files(ROOT_DIR)

    image_stems = set(image_files.keys())
    json_stems = set(json_files.keys())

    matched_stems = image_stems & json_stems

    unmatched_images = sorted(image_stems - json_stems)
    unmatched_jsons = sorted(json_stems - image_stems)

    print("\n========== BEFORE CLEANING ==========")
    print(f"Images before        : {before_image_count}")
    print(f"JSON files before    : {before_json_count}")
    print(f"Total files before   : {before_image_count + before_json_count}")
    print("=====================================")

    print("\n========== MATCH SUMMARY ==========")
    print(f"Matched pairs        : {len(matched_stems)}")
    print(f"Unmatched images     : {len(unmatched_images)}")
    print(f"Unmatched JSON files : {len(unmatched_jsons)}")
    print("===================================")

    if unmatched_images:
        print("\n[UNMATCHED IMAGES - will delete]")
        for stem in unmatched_images:
            print(image_files[stem])

    if unmatched_jsons:
        print("\n[UNMATCHED JSONS - will delete]")
        for stem in unmatched_jsons:
            print(json_files[stem])

    if DRY_RUN:
        print("\n[DRY RUN] Hiçbir dosya silinmedi.")
        print("Silmek için DRY_RUN = False yapıp tekrar çalıştır.")

        print("\n========== DRY RUN ESTIMATE ==========")
        print(f"Images after estimate      : {before_image_count - len(unmatched_images)}")
        print(f"JSON files after estimate  : {before_json_count - len(unmatched_jsons)}")
        print(f"Total files after estimate : {(before_image_count + before_json_count) - (len(unmatched_images) + len(unmatched_jsons))}")
        print("======================================")

        return

    deleted_images = 0
    deleted_jsons = 0

    for stem in unmatched_images:
        path = image_files[stem]
        try:
            path.unlink()
            deleted_images += 1
            print(f"[DELETED IMAGE] {path}")
        except Exception as e:
            print(f"[ERROR] Could not delete image {path}: {e}")

    for stem in unmatched_jsons:
        path = json_files[stem]
        try:
            path.unlink()
            deleted_jsons += 1
            print(f"[DELETED JSON] {path}")
        except Exception as e:
            print(f"[ERROR] Could not delete json {path}: {e}")

    after_image_count, after_json_count = count_all_relevant_files(ROOT_DIR)

    print("\n========== AFTER CLEANING ==========")
    print(f"Images after        : {after_image_count}")
    print(f"JSON files after    : {after_json_count}")
    print(f"Total files after   : {after_image_count + after_json_count}")
    print("====================================")

    print("\n========== DELETED ==========")
    print(f"Deleted images      : {deleted_images}")
    print(f"Deleted jsons       : {deleted_jsons}")
    print(f"Total deleted       : {deleted_images + deleted_jsons}")
    print("=============================")

    print("\n========== CHECK ==========")
    print(f"Before total        : {before_image_count + before_json_count}")
    print(f"After total         : {after_image_count + after_json_count}")
    print(f"Difference          : {(before_image_count + before_json_count) - (after_image_count + after_json_count)}")
    print("===========================")


if __name__ == "__main__":
    main()

### cropper

In [ ]:
import json
from pathlib import Path
from PIL import Image

# =========================
# Ana klasör yolları
# =========================
root_dir = Path(r"C:\Users\askin\OneDrive\Masaüstü\dataset_clean\sahneler\S8")

# Çıktı klasörü
output_dir = Path(r"C:\Users\askin\OneDrive\Masaüstü\dataset_clean\s8_cropped")
output_dir.mkdir(parents=True, exist_ok=True)

# =========================
# Çok küçük bbox filtreleri
# =========================
MIN_WIDTH = 5
MIN_HEIGHT = 5

# =========================
# ID seçimi
# character_id kullanmak için: "character"
# semantic_id kullanmak için : "semantic"
# =========================
ID_MODE = "character"

# =========================
# Sayaçlar
# =========================
total_scene_folders = 0
total_json = 0
total_saved = 0
total_skipped_small = 0
total_skipped_invalid = 0
total_missing_image = 0
total_error = 0
total_skipped_black = 0

# =========================
# sahneler altındaki S* klasörlerini gez
# =========================
scene_dirs = sorted([p for p in root_dir.glob("S*") if p.is_dir()])

print(f"[INFO] Bulunan sahne klasörü sayısı: {len(scene_dirs)}")

for input_dir in scene_dirs:
    total_scene_folders += 1
    json_files = sorted(input_dir.glob("*.json"))

    print(f"\n[SCENE] {input_dir.name} | JSON sayısı: {len(json_files)}")

    for json_path in json_files:
        total_json += 1

        try:
            # JSON oku
            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # JSON içindeki image_file varsa onu kullan
            image_name = data.get("image_file", json_path.with_suffix(".jpg").name)
            image_path = input_dir / image_name

            # Eğer image_file bulunamazsa, JSON ile aynı isimli jpg dene
            if not image_path.exists():
                image_path = json_path.with_suffix(".jpg")

            # Görsel yoksa geç
            if not image_path.exists():
                print(f"[MISSING IMAGE] scene={input_dir.name} | json={json_path.name}")
                total_missing_image += 1
                continue

            # Görseli aç
            img = Image.open(image_path).convert("RGB")
            img_w, img_h = img.size

            # Metadata
            camera_id = int(data["camera_id"])
            sequence_id = int(data["sequence_id"])
            frame_id = int(data["frame_id"])
            variant_id = int(data["variant_id"])

            boxes = data["annotations"]["boxes"]

            for box in boxes:
                raw_character_id = box.get("character_id", "NO_CHARACTER_ID")
                semantic_id = box.get("semantic_id", None)

                # =========================
                # Dosya adındaki insan ID seçimi
                # =========================
                if ID_MODE == "character":
                    id_num = int(str(raw_character_id).replace("_", ""))
                elif ID_MODE == "semantic":
                    id_num = int(semantic_id)
                else:
                    raise ValueError("ID_MODE sadece 'character' veya 'semantic' olabilir.")

                x_min = int(box["x_min"])
                y_min = int(box["y_min"])
                x_max = int(box["x_max"])
                y_max = int(box["y_max"])

                # Görsel sınırları içine al
                x_min = max(0, min(x_min, img_w))
                y_min = max(0, min(y_min, img_h))
                x_max = max(0, min(x_max, img_w))
                y_max = max(0, min(y_max, img_h))

                width = x_max - x_min
                height = y_max - y_min

                # Geçersiz bbox kontrolü
                if width <= 0 or height <= 0:
                    print(
                        f"[SKIP INVALID] scene={input_dir.name} | "
                        f"file={json_path.name} | "
                        f"character_id={raw_character_id} | "
                        f"semantic_id={semantic_id} | "
                        f"bbox=({x_min}, {y_min}, {x_max}, {y_max}) | "
                        f"size={width}x{height}"
                    )
                    total_skipped_invalid += 1
                    continue

                # Aşırı ince / anlamsız bbox kontrolü
                if width < MIN_WIDTH or height < MIN_HEIGHT:
                    print(
                        f"[SKIP TOO SMALL] scene={input_dir.name} | "
                        f"file={json_path.name} | "
                        f"character_id={raw_character_id} | "
                        f"semantic_id={semantic_id} | "
                        f"bbox=({x_min}, {y_min}, {x_max}, {y_max}) | "
                        f"size={width}x{height}"
                    )
                    total_skipped_small += 1
                    continue

                crop = img.crop((x_min, y_min, x_max, y_max))

                # =========================
                # Tamamen siyah crop kontrolü
                # =========================
                extrema = crop.getextrema()
                # RGB için örnek siyah: ((0, 0), (0, 0), (0, 0))

                is_fully_black = all(channel_max == 0 for channel_min, channel_max in extrema)

                if is_fully_black:
                    total_skipped_black += 1
                    print(
                        f"[SKIP FULL BLACK] scene={input_dir.name} | "
                        f"file={json_path.name} | "
                        f"bbox=({x_min}, {y_min}, {x_max}, {y_max}) | "
                        f"size={width}x{height} | "
                        f"extrema={extrema}"
                    )
                    continue

                # İsim formatı:
                out_name = (
                    f"{id_num:04d}"
                    f"_c{camera_id}s{sequence_id}"
                    f"_{frame_id:06d}"
                    f"_{variant_id:02d}.jpg"
                )

                out_path = output_dir / out_name

                # Aynı isim varsa üstüne yazmamak için suffix ekle
                if out_path.exists():
                    stem = out_path.stem
                    suffix = out_path.suffix
                    i = 1

                    while True:
                        new_out_path = output_dir / f"{stem}_dup{i}{suffix}"
                        if not new_out_path.exists():
                            out_path = new_out_path
                            break
                        i += 1

                crop.save(out_path, quality=95)
                total_saved += 1

                print(
                    f"[OK] {out_path.name} kaydedildi | "
                    f"scene={input_dir.name} | "
                    f"source={image_path.name} | "
                    f"character_id={raw_character_id} | "
                    f"semantic_id={semantic_id} | "
                    f"bbox=({x_min}, {y_min}, {x_max}, {y_max}) | "
                    f"size={width}x{height}"
                )

        except Exception as e:
            print(f"[ERROR] scene={input_dir.name} | json={json_path.name} | hata={e}")
            total_error += 1


# =========================
# Özet
# =========================
print("\n========== ÖZET ==========")
print(f"İşlenen sahne klasörü     : {total_scene_folders}")
print(f"İşlenen JSON sayısı       : {total_json}")
print(f"Kaydedilen crop sayısı    : {total_saved}")
print(f"Çok küçük diye atlanan    : {total_skipped_small}")
print(f"Geçersiz bbox atlanan     : {total_skipped_invalid}")
print(f"Görsel bulunamayan JSON   : {total_missing_image}")
print(f"Hata alınan JSON          : {total_error}")
print(f"Tam siyah diye atlanan    : {total_skipped_black}")
print("==========================")

### variant sampling (5)

In [ ]:
import random
import shutil
from pathlib import Path
from collections import defaultdict

# =========================
# Klasör yolları
# =========================
input_dir = Path(r"C:\Users\askin\OneDrive\Masaüstü\dataset_clean\all_scenes_crops")

output_dir = Path(r"C:\Users\askin\OneDrive\Masaüstü\dataset_clean\sampled_crops")
output_dir.mkdir(parents=True, exist_ok=True)

# =========================
# Ayarlar
# =========================
STEP = 5
SEED = 42

random.seed(SEED)

# =========================
# Dosyaları gruplama
# Format:
# 0018_c3s8_000001_00.jpg
# =========================
groups = defaultdict(list)

image_files = sorted(input_dir.glob("*.jpg"))

for img_path in image_files:
    stem = img_path.stem
    parts = stem.split("_")

    if len(parts) != 4:
        print(f"[SKIP NAME FORMAT] {img_path.name}")
        continue

    person_or_semantic_id = parts[0]   # 0018
    cam_scene = parts[1]               # c3s8
    frame_id = parts[2]                # 000001
    variant_id = parts[3]              # 00

    try:
        variant_num = int(variant_id)
    except ValueError:
        print(f"[SKIP VARIANT ERROR] {img_path.name}")
        continue

    group_key = f"{person_or_semantic_id}_{cam_scene}_{frame_id}"
    groups[group_key].append((variant_num, img_path))


# =========================
# Her grupta 5'lik aralıktan 1 random seç
# =========================
total_groups = 0
total_selected = 0
total_copied = 0

for group_key, items in groups.items():
    total_groups += 1

    items = sorted(items, key=lambda x: x[0])

    selected_files = []

    for i in range(0, len(items), STEP):
        chunk = items[i:i + STEP]

        if not chunk:
            continue

        selected_variant, selected_path = random.choice(chunk)
        selected_files.append(selected_path)

    for selected_path in selected_files:
        out_path = output_dir / selected_path.name

        if out_path.exists():
            stem = out_path.stem
            suffix = out_path.suffix
            dup_idx = 1

            while True:
                new_out_path = output_dir / f"{stem}_dup{dup_idx}{suffix}"
                if not new_out_path.exists():
                    out_path = new_out_path
                    break
                dup_idx += 1

        shutil.copy2(selected_path, out_path)
        total_copied += 1

    total_selected += len(selected_files)

    print(
        f"[OK] group={group_key} | "
        f"total={len(items)} | "
        f"selected={len(selected_files)}"
    )


print("\n========== ÖZET ==========")
print(f"Toplam crop sayısı       : {len(image_files)}")
print(f"Toplam grup sayısı       : {total_groups}")
print(f"Seçilen crop sayısı      : {total_selected}")
print(f"Kopyalanan crop sayısı   : {total_copied}")
print(f"Output klasörü           : {output_dir}")
print("==========================")

### check

In [ ]:
from pathlib import Path
from PIL import Image
from collections import Counter, defaultdict
import re

# =========================
# Kontrol edilecek klasör
# =========================
input_dir = Path(r"C:\Users\askin\OneDrive\Masaüstü\dataset_clean\sampled_crops")

# Rapor klasörü
report_dir = Path(r"C:\Users\askin\OneDrive\Masaüstü\dataset_clean\preprocess_reports")
report_dir.mkdir(parents=True, exist_ok=True)

# =========================
# Beklenen dosya formatı
# Örnek:
# 0018_c3s8_000001_00.jpg
# =========================
filename_pattern = re.compile(
    r"^(?P<id>\d{4})_c(?P<camera>\d+)s(?P<scene>\d+)_(?P<frame>\d{6})_(?P<variant>\d{2})(?:_dup\d+)?\.jpg$"
)

# =========================
# Crop size eşikleri
# Bunlar sadece raporlama için.
# Dosya silmez.
# =========================
MIN_WIDTH_WARN = 10
MIN_HEIGHT_WARN = 10
MAX_RATIO_WARN = 8.0  # Çok ince/uzun crop kontrolü

image_files = sorted(input_dir.glob("*.jpg"))

id_counter = Counter()
bad_name_files = []
size_records = []
suspicious_size_files = []
read_error_files = []

camera_counter = Counter()
scene_counter = Counter()
variant_counter = Counter()

for img_path in image_files:
    match = filename_pattern.match(img_path.name)

    if not match:
        bad_name_files.append(img_path.name)
        continue

    file_id = match.group("id")
    camera_id = match.group("camera")
    scene_id = match.group("scene")
    variant_id = match.group("variant")

    id_counter[file_id] += 1
    camera_counter[camera_id] += 1
    scene_counter[scene_id] += 1
    variant_counter[variant_id] += 1

    try:
        with Image.open(img_path) as img:
            w, h = img.size

        ratio = max(w / h, h / w) if w > 0 and h > 0 else 999

        size_records.append((img_path.name, file_id, w, h, ratio))

        if w < MIN_WIDTH_WARN or h < MIN_HEIGHT_WARN or ratio > MAX_RATIO_WARN:
            suspicious_size_files.append((img_path.name, file_id, w, h, ratio))

    except Exception as e:
        read_error_files.append((img_path.name, str(e)))


# =========================
# 1) ID distribution raporu
# =========================
id_report_path = report_dir / "id_distribution.txt"

with open(id_report_path, "w", encoding="utf-8") as f:
    f.write("ID DISTRIBUTION REPORT\n")
    f.write("======================\n\n")
    f.write(f"Total valid images: {sum(id_counter.values())}\n")
    f.write(f"Unique ID count   : {len(id_counter)}\n\n")

    f.write("ID image counts:\n")
    f.write("----------------\n")

    for file_id, count in id_counter.most_common():
        f.write(f"{file_id}: {count}\n")

    f.write("\nIDs with very few images <= 3:\n")
    f.write("------------------------------\n")

    few_ids = [(file_id, count) for file_id, count in id_counter.items() if count <= 3]

    if few_ids:
        for file_id, count in sorted(few_ids, key=lambda x: x[1]):
            f.write(f"{file_id}: {count}\n")
    else:
        f.write("None\n")


# =========================
# 2) Bad filename raporu
# =========================
bad_name_report_path = report_dir / "bad_name_files.txt"

with open(bad_name_report_path, "w", encoding="utf-8") as f:
    f.write("BAD NAME FILES REPORT\n")
    f.write("=====================\n\n")
    f.write(f"Total jpg files checked : {len(image_files)}\n")
    f.write(f"Bad filename count      : {len(bad_name_files)}\n\n")

    if bad_name_files:
        for name in bad_name_files:
            f.write(name + "\n")
    else:
        f.write("No bad filename found.\n")


# =========================
# 3) Crop size raporu
# =========================
size_report_path = report_dir / "crop_size_report.txt"

with open(size_report_path, "w", encoding="utf-8") as f:
    f.write("CROP SIZE REPORT\n")
    f.write("================\n\n")
    f.write(f"Total valid size records : {len(size_records)}\n")
    f.write(f"Suspicious size count    : {len(suspicious_size_files)}\n")
    f.write(f"Read error count         : {len(read_error_files)}\n\n")

    if size_records:
        widths = [r[2] for r in size_records]
        heights = [r[3] for r in size_records]
        ratios = [r[4] for r in size_records]

        f.write("General statistics:\n")
        f.write("-------------------\n")
        f.write(f"Min width   : {min(widths)}\n")
        f.write(f"Max width   : {max(widths)}\n")
        f.write(f"Min height  : {min(heights)}\n")
        f.write(f"Max height  : {max(heights)}\n")
        f.write(f"Min ratio   : {min(ratios):.2f}\n")
        f.write(f"Max ratio   : {max(ratios):.2f}\n\n")

    f.write("Suspicious crop sizes:\n")
    f.write("----------------------\n")

    if suspicious_size_files:
        for name, file_id, w, h, ratio in suspicious_size_files:
            f.write(f"{name} | id={file_id} | size={w}x{h} | ratio={ratio:.2f}\n")
    else:
        f.write("No suspicious crop size found.\n")

    f.write("\nRead errors:\n")
    f.write("------------\n")

    if read_error_files:
        for name, err in read_error_files:
            f.write(f"{name} | error={err}\n")
    else:
        f.write("No read errors.\n")


# =========================
# Ek küçük özet: camera / scene / variant dağılımı
# =========================
extra_report_path = report_dir / "camera_scene_variant_distribution.txt"

with open(extra_report_path, "w", encoding="utf-8") as f:
    f.write("CAMERA / SCENE / VARIANT DISTRIBUTION\n")
    f.write("=====================================\n\n")

    f.write("Camera distribution:\n")
    f.write("--------------------\n")
    for cam, count in sorted(camera_counter.items(), key=lambda x: int(x[0])):
        f.write(f"c{cam}: {count}\n")

    f.write("\nScene distribution:\n")
    f.write("-------------------\n")
    for scene, count in sorted(scene_counter.items(), key=lambda x: int(x[0])):
        f.write(f"s{scene}: {count}\n")

    f.write("\nVariant distribution:\n")
    f.write("---------------------\n")
    for variant, count in sorted(variant_counter.items(), key=lambda x: int(x[0])):
        f.write(f"{variant}: {count}\n")


# =========================
# Konsol özeti
# =========================
print("\n========== CHECK ÖZET ==========")
print(f"Kontrol edilen jpg sayısı       : {len(image_files)}")
print(f"Formatı doğru dosya sayısı      : {sum(id_counter.values())}")
print(f"Formatı bozuk dosya sayısı      : {len(bad_name_files)}")
print(f"Unique ID sayısı                : {len(id_counter)}")
print(f"Şüpheli crop size sayısı        : {len(suspicious_size_files)}")
print(f"Okuma hatası sayısı             : {len(read_error_files)}")
print("--------------------------------")
print(f"ID raporu                       : {id_report_path}")
print(f"Bozuk isim raporu               : {bad_name_report_path}")
print(f"Crop size raporu                : {size_report_path}")
print(f"Camera/scene/variant raporu     : {extra_report_path}")
print("================================")